# 06 - Huấn luyện Random Forest trên Spark MLlib

Huấn luyện `RandomForestRegressor` để dự báo `log_doanh_so = log1p(sales)` (biến
đổi log để giảm lệch phân phối, phù hợp thước đo RMSLE gốc của cuộc thi Kaggle).

**Chia tập theo thời gian, không random split**: vì đây là bài toán chuỗi thời
gian, validation phải mô phỏng đúng tình huống dự báo thật — giữ lại **16 ngày
cuối cùng của train** (bằng đúng độ dài `HORIZON` của test) làm tập kiểm định,
huấn luyện trên phần còn lại.

In [ ]:
import sys
import time
from datetime import timedelta
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

THU_MUC_HIEN_TAI = Path.cwd()
GOC = THU_MUC_HIEN_TAI.parent if THU_MUC_HIEN_TAI.name == "notebooks" else THU_MUC_HIEN_TAI
sys.path.append(str(GOC / "src"))

from cau_hinh import DATA_PROCESSED, OUT_TABLES, OUT_FIGURES, OUT_MODELS, SEED, HORIZON, tao_thu_muc
from spark_session import tao_spark
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, Imputer, VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

tao_thu_muc()
spark = tao_spark("06_RandomForest")

df_train = spark.read.parquet(str(DATA_PROCESSED / "train_dac_trung"))
df_test = spark.read.parquet(str(DATA_PROCESSED / "test_dac_trung"))
print("So dong train_dac_trung:", df_train.count(), "| test_dac_trung:", df_test.count())

MAU_CHINH = "#2a78d6"
MAU_LUOI = "#e1e0d9"
MAU_TRUC = "#c3c2b7"
MAU_CHU_PHU = "#52514e"
MAU_CHU_CHINH = "#0b0b0b"

## 1. Chia tập theo thời gian

In [ ]:
ngay_max_train = df_train.select(F.max("date")).first()[0]
ngay_cutoff = ngay_max_train - timedelta(days=HORIZON - 1)
print(f"Ngay max train: {ngay_max_train} | HORIZON = {HORIZON} ngay | Cutoff kiem dinh: {ngay_cutoff}")

df_huan_luyen = df_train.filter(F.col("date") < ngay_cutoff)
df_kiem_dinh = df_train.filter(F.col("date") >= ngay_cutoff)
print("So dong huan luyen:", df_huan_luyen.count(), "| kiem dinh:", df_kiem_dinh.count())
assert df_kiem_dinh.select(F.countDistinct("date")).first()[0] == HORIZON

## 2. Pipeline: mã hoá phân loại + điền khuyết + vector hoá + Random Forest

Cột phân loại (`family`, `city`, `state`, `type`) qua `StringIndexer` — **không**
one-hot, vì cây quyết định trong Spark MLlib tự đọc metadata để tách nhánh theo
danh mục, one-hot chỉ cần thiết cho mô hình tuyến tính. Các cột lag/rolling còn
null (do lịch sử chưa đủ, xem notebook 05) được điền bằng trung bình cột
(`Imputer`, khớp trên riêng tập huấn luyện).

In [ ]:
COT_CAN_IMPUTE = ["lag_7", "lag_14", "lag_28", "tb_truot_7", "tb_truot_28", "do_lech_chuan_truot_28"]
COT_SO_AN_TOAN = ["onpromotion", "cluster", "gia_dau", "nam", "thang", "ngay_trong_thang", "thu", "tuan_trong_nam"]
COT_BOOL = ["la_ngay_le", "la_su_kien_dac_biet", "la_cuoi_tuan"]
COT_PHAN_LOAI = ["family", "city", "state", "type"]

indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in COT_PHAN_LOAI]
imputer = Imputer(strategy="mean", inputCols=COT_CAN_IMPUTE, outputCols=[f"{c}_dd" for c in COT_CAN_IMPUTE])

COT_DAU_VAO = (COT_SO_AN_TOAN + [f"{c}_dd" for c in COT_CAN_IMPUTE] + COT_BOOL
               + [f"{c}_idx" for c in COT_PHAN_LOAI])
assembler = VectorAssembler(inputCols=COT_DAU_VAO, outputCol="dac_trung", handleInvalid="skip")

# family co 33 gia tri phan biet -> maxBins phai >= 33
rf = RandomForestRegressor(featuresCol="dac_trung", labelCol="log_doanh_so", predictionCol="du_bao_log",
                            numTrees=60, maxDepth=12, maxBins=64, seed=SEED, subsamplingRate=0.7)

pipeline = Pipeline(stages=indexers + [imputer, assembler, rf])

## 3. Huấn luyện trên tập huấn luyện (trước ngày cutoff)

In [ ]:
t0 = time.time()
mo_hinh = pipeline.fit(df_huan_luyen)
thoi_gian_huan_luyen = round(time.time() - t0, 1)
print("Thoi gian huan luyen (s):", thoi_gian_huan_luyen)

## 4. Đánh giá trên tập kiểm định (16 ngày cuối) so với baseline mùa vụ ngây thơ

Baseline: dự báo doanh số ngày hôm nay bằng đúng doanh số **cùng thứ, 4 tuần
trước** (`lag_28`) — một baseline mùa vụ hợp lý, không cần huấn luyện, để biết
Random Forest có thực sự học được gì hơn ngoài việc lặp lại mùa vụ hay không.

In [ ]:
du_bao = mo_hinh.transform(df_kiem_dinh)
du_bao = du_bao.withColumn("sales_du_bao", F.greatest(F.expm1("du_bao_log"), F.lit(0.0)))

rmse_log = RegressionEvaluator(labelCol="log_doanh_so", predictionCol="du_bao_log",
                                metricName="rmse").evaluate(du_bao)
rmse_goc = RegressionEvaluator(labelCol="sales", predictionCol="sales_du_bao",
                                metricName="rmse").evaluate(du_bao)
mae_goc = RegressionEvaluator(labelCol="sales", predictionCol="sales_du_bao",
                               metricName="mae").evaluate(du_bao)
rmsle_rf = du_bao.select(
    F.sqrt(F.avg(F.pow(F.log1p("sales_du_bao") - F.log1p("sales"), 2)))
).first()[0]

baseline = df_kiem_dinh.withColumn("sales_baseline", F.coalesce(F.col("lag_28"), F.lit(0.0)))
rmsle_baseline = baseline.select(
    F.sqrt(F.avg(F.pow(F.log1p("sales_baseline") - F.log1p("sales"), 2)))
).first()[0]

cai_thien = 100 * (rmsle_baseline - rmsle_rf) / rmsle_baseline
print(f"RMSE (log)              = {rmse_log:.4f}")
print(f"RMSE (thang doanh so)   = {rmse_goc:.2f}")
print(f"MAE  (thang doanh so)   = {mae_goc:.2f}")
print(f"RMSLE Random Forest     = {rmsle_rf:.4f}")
print(f"RMSLE baseline (lag_28) = {rmsle_baseline:.4f}")
print(f"Cai thien so voi baseline: {cai_thien:.1f}%")

bang_danh_gia = pd.DataFrame([
    {"mo_hinh": "RandomForest", "rmse_log": rmse_log, "rmse_goc": rmse_goc, "mae_goc": mae_goc, "rmsle": rmsle_rf},
    {"mo_hinh": "Baseline_mua_vu_lag28", "rmse_log": None, "rmse_goc": None, "mae_goc": None, "rmsle": rmsle_baseline},
])
bang_danh_gia.to_csv(OUT_TABLES / "danh_gia_random_forest.csv", index=False, encoding="utf-8-sig")
bang_danh_gia

## 5. Mức độ quan trọng của đặc trưng

In [ ]:
rf_model = mo_hinh.stages[-1]
do_quan_trong = pd.DataFrame(
    list(zip(COT_DAU_VAO, rf_model.featureImportances.toArray())),
    columns=["dac_trung", "do_quan_trong"],
).sort_values("do_quan_trong", ascending=False).reset_index(drop=True)
do_quan_trong.to_csv(OUT_TABLES / "do_quan_trong_dac_trung.csv", index=False, encoding="utf-8-sig")

top12 = do_quan_trong.head(12).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top12["dac_trung"], top12["do_quan_trong"], color=MAU_CHINH)
ax.set_xlabel("Do quan trong (Gini importance)", color=MAU_CHU_PHU)
ax.set_title("Top 12 dac trung quan trong nhat - Random Forest", color=MAU_CHU_CHINH, loc="left")
ax.grid(axis="x", color=MAU_LUOI, linewidth=1, zorder=0)
ax.set_axisbelow(True)
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
for spine in ["left", "bottom"]:
    ax.spines[spine].set_color(MAU_TRUC)
ax.tick_params(colors=MAU_CHU_PHU)
fig.tight_layout()
fig.savefig(OUT_FIGURES / "random_forest_do_quan_trong.png", dpi=150)
plt.show()

In [ ]:
duong_dan_mo_hinh_kiem_dinh = OUT_MODELS / "random_forest_pipeline_kiem_dinh"
mo_hinh.write().overwrite().save(str(duong_dan_mo_hinh_kiem_dinh))
print("Da luu mo hinh (chi huan luyen tren du lieu truoc cutoff, dung cho notebook 07):",
      duong_dan_mo_hinh_kiem_dinh)

## 6. Huấn luyện lại trên toàn bộ train và dự báo tập test chính thức

Sau khi đã kiểm chứng kiến trúc/tham số ở bước 4, huấn luyện lại trên **toàn bộ**
`train_dac_trung` (bao gồm cả 16 ngày vừa dùng kiểm định) để tận dụng hết dữ liệu
trước khi dự báo 16 ngày thật của `test_dac_trung` — đúng thông lệ Kaggle.

In [ ]:
t0 = time.time()
mo_hinh_cuoi = pipeline.fit(df_train)
print("Thoi gian huan luyen lai tren toan bo train (s):", round(time.time() - t0, 1))

du_bao_test = mo_hinh_cuoi.transform(df_test)
du_bao_test = du_bao_test.withColumn("sales", F.greatest(F.expm1("du_bao_log"), F.lit(0.0)))

ket_qua_nop = du_bao_test.select("id", "sales").orderBy("id").toPandas()
print(ket_qua_nop.describe())
assert len(ket_qua_nop) == df_test.count()
assert ket_qua_nop["sales"].isna().sum() == 0
assert (ket_qua_nop["sales"] >= 0).all()

ket_qua_nop.to_csv(OUT_TABLES / "du_bao_test.csv", index=False, encoding="utf-8-sig")
print("Da luu:", OUT_TABLES / "du_bao_test.csv")

## 7. Lưu mô hình

In [ ]:
duong_dan_mo_hinh = OUT_MODELS / "random_forest_pipeline"
mo_hinh_cuoi.write().overwrite().save(str(duong_dan_mo_hinh))
print("Da luu mo hinh:", duong_dan_mo_hinh)

In [ ]:
spark.stop()